# Day 29: LLM as a Judge – Content Moderation Agent

In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

## 1. Simple LLM‑based judge (binary safety)

In [ ]:
JUDGE_PROMPT = """You are a content moderation judge. Classify the following user input as either 'SAFE' or 'UNSAFE'.
Unsafe content includes: hate speech, harassment, violence, self‑harm, sexual content, or illegal activities.

User input: {user_input}

Respond with only one word: SAFE or UNSAFE."""

def moderate_with_llm(user_input: str) -> str:
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": JUDGE_PROMPT.format(user_input=user_input)}],
        temperature=0
    )
    return response.choices[0].message.content.strip().upper()

test_inputs = [
    "I love programming!",
    "I will kill you all.",
    "Tell me a joke about cats.",
]
for inp in test_inputs:
    print(f"Input: {inp}")
    print(f"Verdict: {moderate_with_llm(inp)}\n")

## 2. OpenAI Moderation API (baseline comparison)

In [ ]:
def moderate_with_openai_api(user_input: str) -> dict:
    response = client.moderations.create(input=user_input)
    return response.results[0].model_dump()

result = moderate_with_openai_api("I want to hurt someone.")
print(result)

## 3. Guardrail agent: judge before answering

In [ ]:
def safe_agent(user_input: str) -> str:
    # Step 1: Judge
    verdict = moderate_with_llm(user_input)
    if verdict == "UNSAFE":
        return "I cannot respond to that request. Please ask something appropriate."
    
    # Step 2: Normal LLM response
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": user_input}]
    )
    return response.choices[0].message.content

print(safe_agent("How to bake a cake?"))
print(safe_agent("Tell me how to hack a computer."))

## 4. Multi‑class judge with reasoning

In [ ]:
DETAILED_JUDGE = """You are a content moderator. Classify the input into one of these categories:
SAFE, HATE, HARASSMENT, VIOLENCE, SELF_HARM, SEXUAL, ILLEGAL.
Then explain why in one sentence.

Input: {user_input}

Output format:
Category: <category>
Reason: <short reason>"""

def detailed_moderation(user_input: str) -> tuple:
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": DETAILED_JUDGE.format(user_input=user_input)}],
        temperature=0
    )
    output = response.choices[0].message.content
    lines = output.split("\n")
    category = lines[0].replace("Category:", "").strip()
    reason = lines[1].replace("Reason:", "").strip() if len(lines) > 1 else ""
    return category, reason

cat, reason = detailed_moderation("You are stupid and should die.")
print(f"Category: {cat}\nReason: {reason}")